# Topological Time Irreversibility in Financial Time Series  
## Exploratory notebook for a finance paper

This notebook implements a complete exploratory pipeline to study **temporal topological asymmetry** in financial time series using data from **Yahoo Finance**.

### Central idea
For each rolling window, we compare the persistent-homology structure of:

- the **original** time series segment (forward),
- the **time-reversed** segment (backward).

The goal is to quantify **topological irreversibility** through distances between persistence diagrams and related summaries.

### What this notebook does
1. Downloads daily market data from Yahoo Finance.
2. Builds log-return series.
3. Computes **Takens embeddings**.
4. Extracts persistence diagrams with **Ripser**.
5. Defines a **Topological Time Irreversibility (TTI)** index.
6. Computes rolling TTI by asset.
7. Compares TTI against rolling volatility.
8. Builds null models using surrogates.
9. Produces figures and exportable tables for paper writing.

### Suggested assets
- Equity indices: `^GSPC`, `^IXIC`, `^IBEX`, `^N225`
- FX: `EURUSD=X`, `JPY=X`, `USDCOP=X`
- Crypto: `BTC-USD`, `ETH-USD`
- Commodities: `GC=F`, `CL=F`

You can change them below.

In [ ]:

# If you are running this notebook in Google Colab, uncomment:
# !pip -q install yfinance ripser persim scikit-learn

In [ ]:

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional

import yfinance as yf

from ripser import ripser
from persim import plot_diagrams, wasserstein

from sklearn.linear_model import LinearRegression
from scipy.signal import periodogram
from scipy.stats import zscore, spearmanr

## 1. Configuration
Set the universe of assets and the methodological parameters.

In [ ]:

# ----------------------------
# USER CONFIGURATION
# ----------------------------

TICKERS = {
    "SP500": "^GSPC",
    "NASDAQ": "^IXIC",
    "IBEX35": "^IBEX",
    "NIKKEI225": "^N225",
    "EURUSD": "EURUSD=X",
    "USDJPY": "JPY=X",
    "USDCOP": "USDCOP=X",
    "BTC": "BTC-USD",
    "ETH": "ETH-USD",
    "GOLD": "GC=F",
    "WTI": "CL=F",
}

START_DATE = "2017-01-01"
END_DATE = None  # None means up to latest available date

PRICE_FIELD = "Close"   # with auto_adjust=True, Close is adjusted

# Rolling-window/TDA parameters
WINDOW = 180
STEP = 10
EMBED_DIM = 3
TAU = 2
MAXDIM = 1

# Surrogate testing
N_SURROGATES = 40
SURROGATE_WINDOW_SAMPLE = 8   # number of windows sampled per asset for null analysis

# Volatility benchmark
VOL_WINDOW = 30

# Output folder
OUTPUT_DIR = "tti_outputs"

## 2. Helper functions
We define utilities for downloading data, preprocessing returns, constructing embeddings, computing persistence diagrams, and generating topological irreversibility metrics.

In [ ]:

def ensure_output_dir(path: str):
    import os
    os.makedirs(path, exist_ok=True)

ensure_output_dir(OUTPUT_DIR)


def download_prices(tickers: Dict[str, str],
                    start: str,
                    end: Optional[str] = None,
                    price_field: str = "Close") -> pd.DataFrame:
    """Download price panel from Yahoo Finance."""
    raw = yf.download(
        tickers=list(tickers.values()),
        start=start,
        end=end,
        auto_adjust=True,
        progress=False,
        threads=True,
    )

    if isinstance(raw.columns, pd.MultiIndex):
        px = raw[price_field].copy()
    else:
        px = raw[[price_field]].copy()

    rename_map = {v: k for k, v in tickers.items()}
    px = px.rename(columns=rename_map)
    px = px.sort_index()
    return px


def compute_log_returns(prices: pd.DataFrame) -> pd.DataFrame:
    rets = np.log(prices / prices.shift(1))
    return rets.dropna(how="all")


def takens_embedding(x: np.ndarray, m: int = 3, tau: int = 1) -> np.ndarray:
    """Takens embedding of a 1D series into R^m."""
    x = np.asarray(x, dtype=float)
    n = len(x)
    needed = (m - 1) * tau + 1
    if n < needed:
        raise ValueError("Series too short for requested embedding.")
    rows = n - (m - 1) * tau
    emb = np.column_stack([x[i : i + rows] for i in range(0, m * tau, tau)])
    return emb


def finite_diagram(diagram: np.ndarray) -> np.ndarray:
    """Keep only finite persistence pairs."""
    if diagram.size == 0:
        return np.empty((0, 2))
    mask = np.isfinite(diagram).all(axis=1)
    return diagram[mask]


def persistence_entropy(diagram: np.ndarray) -> float:
    dgm = finite_diagram(diagram)
    if dgm.size == 0:
        return 0.0
    lifetimes = dgm[:, 1] - dgm[:, 0]
    lifetimes = lifetimes[lifetimes > 0]
    if len(lifetimes) == 0:
        return 0.0
    p = lifetimes / lifetimes.sum()
    return float(-(p * np.log(p)).sum())


def total_persistence(diagram: np.ndarray, p: float = 1.0) -> float:
    dgm = finite_diagram(diagram)
    if dgm.size == 0:
        return 0.0
    lifetimes = np.maximum(dgm[:, 1] - dgm[:, 0], 0.0)
    return float(np.sum(lifetimes ** p))


def safe_wasserstein(dgm1: np.ndarray, dgm2: np.ndarray) -> float:
    d1 = finite_diagram(dgm1)
    d2 = finite_diagram(dgm2)
    if d1.size == 0 and d2.size == 0:
        return 0.0
    return float(wasserstein(d1, d2))


def compute_diagrams_from_series(x: np.ndarray,
                                 embed_dim: int = 3,
                                 tau: int = 1,
                                 maxdim: int = 1) -> List[np.ndarray]:
    """Compute persistence diagrams from Takens embedding."""
    emb = takens_embedding(x, m=embed_dim, tau=tau)
    dgms = ripser(emb, maxdim=maxdim)["dgms"]
    return dgms


def zscore_safe(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=float)
    s = x.std(ddof=0)
    if s == 0 or np.isnan(s):
        return x - x.mean()
    return (x - x.mean()) / s


def phase_randomized_surrogate(x: np.ndarray, rng: np.random.Generator) -> np.ndarray:
    """Fourier phase randomization preserving amplitude spectrum approximately."""
    x = np.asarray(x, dtype=float)
    n = len(x)
    x0 = x - np.mean(x)
    fft = np.fft.rfft(x0)
    amplitudes = np.abs(fft)
    phases = np.angle(fft)

    random_phases = rng.uniform(0, 2 * np.pi, size=len(fft))
    random_phases[0] = phases[0]
    if n % 2 == 0:
        random_phases[-1] = phases[-1]

    fft_new = amplitudes * np.exp(1j * random_phases)
    surrogate = np.fft.irfft(fft_new, n=n)
    surrogate = surrogate + np.mean(x)
    return surrogate


def shuffled_surrogate(x: np.ndarray, rng: np.random.Generator) -> np.ndarray:
    y = np.array(x, copy=True)
    rng.shuffle(y)
    return y

In [ ]:

@dataclass
class TTIResult:
    dgm_forward: List[np.ndarray]
    dgm_backward: List[np.ndarray]
    wasserstein_h0: float
    wasserstein_h1: float
    entropy_diff_h0: float
    entropy_diff_h1: float
    total_persistence_diff_h0: float
    total_persistence_diff_h1: float
    tti_composite: float


def compute_tti_window(x: np.ndarray,
                       embed_dim: int = 3,
                       tau: int = 1,
                       maxdim: int = 1,
                       normalize: bool = True) -> TTIResult:
    """Compute topological time irreversibility metrics for one window."""
    x = np.asarray(x, dtype=float)
    if normalize:
        x = zscore_safe(x)

    x_f = x
    x_b = x[::-1]

    dgm_f = compute_diagrams_from_series(x_f, embed_dim=embed_dim, tau=tau, maxdim=maxdim)
    dgm_b = compute_diagrams_from_series(x_b, embed_dim=embed_dim, tau=tau, maxdim=maxdim)

    w0 = safe_wasserstein(dgm_f[0], dgm_b[0])
    w1 = safe_wasserstein(dgm_f[1], dgm_b[1]) if maxdim >= 1 else np.nan

    e0 = abs(persistence_entropy(dgm_f[0]) - persistence_entropy(dgm_b[0]))
    e1 = abs(persistence_entropy(dgm_f[1]) - persistence_entropy(dgm_b[1])) if maxdim >= 1 else np.nan

    tp0 = abs(total_persistence(dgm_f[0]) - total_persistence(dgm_b[0]))
    tp1 = abs(total_persistence(dgm_f[1]) - total_persistence(dgm_b[1])) if maxdim >= 1 else np.nan

    components = [w0, e0, tp0]
    if maxdim >= 1 and np.isfinite(w1):
        components += [w1, e1, tp1]

    composite = float(np.mean(components))

    return TTIResult(
        dgm_forward=dgm_f,
        dgm_backward=dgm_b,
        wasserstein_h0=w0,
        wasserstein_h1=w1,
        entropy_diff_h0=e0,
        entropy_diff_h1=e1,
        total_persistence_diff_h0=tp0,
        total_persistence_diff_h1=tp1,
        tti_composite=composite,
    )

## 3. Download market data

In [ ]:

prices = download_prices(TICKERS, START_DATE, END_DATE, PRICE_FIELD)
returns = compute_log_returns(prices)

print("Prices shape:", prices.shape)
print("Returns shape:", returns.shape)

display(prices.tail())
display(returns.tail())

## 4. Data quality inspection
We retain assets with enough observations to support rolling analysis.

In [ ]:

coverage = returns.notna().sum().sort_values(ascending=False).to_frame("n_obs")
coverage["pct_available"] = coverage["n_obs"] / len(returns)
coverage

In [ ]:

MIN_OBS = WINDOW + 50
valid_assets = coverage.index[coverage["n_obs"] >= MIN_OBS].tolist()

returns_valid = returns[valid_assets].copy()
prices_valid = prices[valid_assets].copy()

print("Assets retained:", valid_assets)

## 5. Quick visual inspection

In [ ]:

fig, ax = plt.subplots(figsize=(12, 5))
(prices_valid / prices_valid.iloc[0]).plot(ax=ax, lw=1)
ax.set_title("Normalized price levels")
ax.set_ylabel("Normalized price")
ax.grid(True, alpha=0.3)
plt.show()

In [ ]:

fig, ax = plt.subplots(figsize=(12, 5))
returns_valid.rolling(30).std().plot(ax=ax, lw=1)
ax.set_title("30-day rolling volatility")
ax.set_ylabel("Volatility")
ax.grid(True, alpha=0.3)
plt.show()

## 6. Single-window prototype
Before running the full rolling pipeline, we inspect one asset and one window to make sure the forward/backward diagrams differ meaningfully.

In [ ]:

ASSET_EXAMPLE = valid_assets[0]
series_example = returns_valid[ASSET_EXAMPLE].dropna().values[:WINDOW]

tti_example = compute_tti_window(
    series_example,
    embed_dim=EMBED_DIM,
    tau=TAU,
    maxdim=MAXDIM,
)

print("Asset:", ASSET_EXAMPLE)
print("Composite TTI:", round(tti_example.tti_composite, 6))
print("Wasserstein H0:", round(tti_example.wasserstein_h0, 6))
print("Wasserstein H1:", round(tti_example.wasserstein_h1, 6))
print("Entropy diff H0:", round(tti_example.entropy_diff_h0, 6))
print("Entropy diff H1:", round(tti_example.entropy_diff_h1, 6))

In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
plot_diagrams(tti_example.dgm_forward, ax=axes[0], show=False)
axes[0].set_title(f"{ASSET_EXAMPLE} - Forward")
plot_diagrams(tti_example.dgm_backward, ax=axes[1], show=False)
axes[1].set_title(f"{ASSET_EXAMPLE} - Backward")
plt.tight_layout()
plt.show()

## 7. Rolling TTI computation
This is the core experiment.

For each asset and each rolling window:
- compute persistent homology of the forward window,
- compute persistent homology of the reversed window,
- evaluate the topological discrepancy.

The resulting time series is our candidate irreversibility signal.

In [ ]:

def rolling_tti_for_series(s: pd.Series,
                           window: int = 180,
                           step: int = 10,
                           embed_dim: int = 3,
                           tau: int = 1,
                           maxdim: int = 1) -> pd.DataFrame:
    s = s.dropna()
    vals = s.values
    idx = s.index

    rows = []
    for start in range(0, len(vals) - window + 1, step):
        end = start + window
        x = vals[start:end]
        date = idx[end - 1]

        try:
            res = compute_tti_window(
                x, embed_dim=embed_dim, tau=tau, maxdim=maxdim, normalize=True
            )
            rows.append({
                "date": date,
                "wasserstein_h0": res.wasserstein_h0,
                "wasserstein_h1": res.wasserstein_h1,
                "entropy_diff_h0": res.entropy_diff_h0,
                "entropy_diff_h1": res.entropy_diff_h1,
                "total_persistence_diff_h0": res.total_persistence_diff_h0,
                "total_persistence_diff_h1": res.total_persistence_diff_h1,
                "tti_composite": res.tti_composite,
            })
        except Exception as e:
            rows.append({
                "date": date,
                "wasserstein_h0": np.nan,
                "wasserstein_h1": np.nan,
                "entropy_diff_h0": np.nan,
                "entropy_diff_h1": np.nan,
                "total_persistence_diff_h0": np.nan,
                "total_persistence_diff_h1": np.nan,
                "tti_composite": np.nan,
            })

    out = pd.DataFrame(rows).set_index("date")
    return out

In [ ]:

rolling_results = {}

for asset in valid_assets:
    print(f"Processing {asset} ...")
    rolling_results[asset] = rolling_tti_for_series(
        returns_valid[asset],
        window=WINDOW,
        step=STEP,
        embed_dim=EMBED_DIM,
        tau=TAU,
        maxdim=MAXDIM,
    )

print("Done.")

In [ ]:

tti_panel = pd.concat(
    {asset: df["tti_composite"] for asset, df in rolling_results.items()},
    axis=1
).sort_index()

tti_panel.tail()

## 8. Plot TTI by asset

In [ ]:

fig, ax = plt.subplots(figsize=(13, 6))
tti_panel.plot(ax=ax, lw=1)
ax.set_title("Rolling topological time irreversibility (TTI)")
ax.set_ylabel("TTI composite")
ax.grid(True, alpha=0.3)
plt.show()

## 9. Relationship with volatility
A basic financial sanity check:
does TTI rise when the market becomes more turbulent?

In [ ]:

rolling_vol = returns_valid.rolling(VOL_WINDOW).std()

# Align at TTI dates
rolling_vol_on_tti = rolling_vol.reindex(tti_panel.index)

corr_rows = []
for asset in valid_assets:
    x = tti_panel[asset]
    y = rolling_vol_on_tti[asset]
    df_xy = pd.concat([x, y], axis=1).dropna()
    if len(df_xy) > 5:
        rho, pval = spearmanr(df_xy.iloc[:, 0], df_xy.iloc[:, 1])
    else:
        rho, pval = np.nan, np.nan
    corr_rows.append({
        "asset": asset,
        "spearman_tti_vs_vol": rho,
        "pvalue": pval,
        "mean_tti": x.mean(),
        "std_tti": x.std(),
        "max_tti": x.max(),
    })

corr_table = pd.DataFrame(corr_rows).sort_values("mean_tti", ascending=False)
corr_table

In [ ]:

fig, ax = plt.subplots(figsize=(12, 5))
corr_table.set_index("asset")["spearman_tti_vs_vol"].sort_values().plot(kind="barh", ax=ax)
ax.set_title("Spearman correlation: TTI vs rolling volatility")
ax.set_xlabel("Correlation")
ax.grid(True, axis="x", alpha=0.3)
plt.show()

## 10. Event-style visualization
Customize the event dates if you want to focus on specific crises.

In [ ]:

EVENT_START = "2020-02-15"
EVENT_END = "2020-06-30"

asset_focus = "SP500" if "SP500" in tti_panel.columns else valid_assets[0]

fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=True)

prices_valid[asset_focus].loc[EVENT_START:EVENT_END].plot(ax=axes[0], lw=1.5)
axes[0].set_title(f"{asset_focus} price around event window")
axes[0].grid(True, alpha=0.3)

tti_panel[asset_focus].loc[EVENT_START:EVENT_END].plot(ax=axes[1], lw=1.5)
axes[1].set_title(f"{asset_focus} TTI around event window")
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 11. Null models and significance
This section is critical for a paper.

We compare observed TTI against:
- a **shuffled surrogate** (destroys temporal ordering),
- a **phase-randomized surrogate** (preserves spectral amplitude approximately while scrambling temporal structure).

Interpretation:
if observed TTI is systematically larger than surrogate TTI, then irreversibility is not just a trivial by-product of variance or spectrum.

In [ ]:

rng = np.random.default_rng(2026)

def sample_window_positions(s: pd.Series, window: int, n_samples: int, step: int = 1):
    s = s.dropna()
    max_start = len(s) - window
    if max_start <= 0:
        return []
    possible = np.arange(0, max_start + 1, step)
    n_take = min(n_samples, len(possible))
    chosen = np.sort(rng.choice(possible, size=n_take, replace=False))
    return chosen.tolist()


def surrogate_tti_distribution(x: np.ndarray,
                               n_surrogates: int = 30,
                               embed_dim: int = 3,
                               tau: int = 1,
                               maxdim: int = 1,
                               rng: Optional[np.random.Generator] = None) -> pd.DataFrame:
    if rng is None:
        rng = np.random.default_rng(0)

    rows = []
    observed = compute_tti_window(x, embed_dim=embed_dim, tau=tau, maxdim=maxdim, normalize=True).tti_composite
    rows.append({"kind": "observed", "tti": observed})

    for _ in range(n_surrogates):
        xs = shuffled_surrogate(x, rng)
        xp = phase_randomized_surrogate(x, rng)

        tti_s = compute_tti_window(xs, embed_dim=embed_dim, tau=tau, maxdim=maxdim, normalize=True).tti_composite
        tti_p = compute_tti_window(xp, embed_dim=embed_dim, tau=tau, maxdim=maxdim, normalize=True).tti_composite

        rows.append({"kind": "shuffled", "tti": tti_s})
        rows.append({"kind": "phase_randomized", "tti": tti_p})

    return pd.DataFrame(rows)

In [ ]:

null_rows = []

for asset in valid_assets:
    s = returns_valid[asset].dropna()
    positions = sample_window_positions(s, WINDOW, SURROGATE_WINDOW_SAMPLE, step=max(1, WINDOW // 6))

    for pos in positions:
        x = s.values[pos:pos+WINDOW]
        date = s.index[pos+WINDOW-1]
        dist = surrogate_tti_distribution(
            x,
            n_surrogates=N_SURROGATES,
            embed_dim=EMBED_DIM,
            tau=TAU,
            maxdim=MAXDIM,
            rng=rng,
        )

        obs_val = dist.loc[dist["kind"] == "observed", "tti"].iloc[0]
        shuf_mean = dist.loc[dist["kind"] == "shuffled", "tti"].mean()
        phase_mean = dist.loc[dist["kind"] == "phase_randomized", "tti"].mean()

        shuf_std = dist.loc[dist["kind"] == "shuffled", "tti"].std(ddof=0)
        phase_std = dist.loc[dist["kind"] == "phase_randomized", "tti"].std(ddof=0)

        z_shuf = (obs_val - shuf_mean) / shuf_std if shuf_std and np.isfinite(shuf_std) else np.nan
        z_phase = (obs_val - phase_mean) / phase_std if phase_std and np.isfinite(phase_std) else np.nan

        null_rows.append({
            "asset": asset,
            "date": date,
            "observed_tti": obs_val,
            "shuffle_mean": shuf_mean,
            "shuffle_std": shuf_std,
            "phase_mean": phase_mean,
            "phase_std": phase_std,
            "z_vs_shuffle": z_shuf,
            "z_vs_phase": z_phase,
        })

null_table = pd.DataFrame(null_rows)
null_table.head()

In [ ]:

null_summary = (
    null_table.groupby("asset")[["observed_tti", "shuffle_mean", "phase_mean", "z_vs_shuffle", "z_vs_phase"]]
    .mean()
    .sort_values("observed_tti", ascending=False)
)
null_summary

In [ ]:

asset_to_plot = null_table["asset"].iloc[0] if len(null_table) else valid_assets[0]
tmp = null_table[null_table["asset"] == asset_to_plot]

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(tmp["date"], tmp["observed_tti"], label="Observed", lw=1.5)
ax.plot(tmp["date"], tmp["shuffle_mean"], label="Shuffle mean", lw=1)
ax.plot(tmp["date"], tmp["phase_mean"], label="Phase-randomized mean", lw=1)
ax.set_title(f"Observed vs surrogate TTI: {asset_to_plot}")
ax.grid(True, alpha=0.3)
ax.legend()
plt.show()

## 12. Cross-asset ranking
This provides a first paper-style cross-sectional comparison.

In [ ]:

asset_classes = {
    "SP500": "Equity",
    "NASDAQ": "Equity",
    "IBEX35": "Equity",
    "NIKKEI225": "Equity",
    "EURUSD": "FX",
    "USDJPY": "FX",
    "USDCOP": "FX",
    "BTC": "Crypto",
    "ETH": "Crypto",
    "GOLD": "Commodity",
    "WTI": "Commodity",
}

ranking = corr_table.copy()
ranking["asset_class"] = ranking["asset"].map(asset_classes).fillna("Other")
ranking = ranking.sort_values(["asset_class", "mean_tti"], ascending=[True, False])
ranking

In [ ]:

class_summary = ranking.groupby("asset_class")[["mean_tti", "std_tti", "max_tti", "spearman_tti_vs_vol"]].mean()
class_summary

In [ ]:

fig, ax = plt.subplots(figsize=(12, 5))
ranking.sort_values("mean_tti", ascending=False).set_index("asset")["mean_tti"].plot(kind="bar", ax=ax)
ax.set_title("Average TTI by asset")
ax.set_ylabel("Mean TTI")
ax.grid(True, axis="y", alpha=0.3)
plt.xticks(rotation=45)
plt.show()

## 13. A compact interpretation helper
This cell produces a draft of textual findings you can reuse when writing the paper.

In [ ]:

def interpret_results(corr_table: pd.DataFrame, null_summary: pd.DataFrame) -> str:
    top_assets = corr_table.sort_values("mean_tti", ascending=False)["asset"].head(3).tolist()
    low_assets = corr_table.sort_values("mean_tti", ascending=True)["asset"].head(3).tolist()

    avg_corr = corr_table["spearman_tti_vs_vol"].mean()
    avg_z_shuffle = null_summary["z_vs_shuffle"].mean() if not null_summary.empty else np.nan
    avg_z_phase = null_summary["z_vs_phase"].mean() if not null_summary.empty else np.nan

    txt = f"""
Main exploratory findings

1. Assets with the highest average topological irreversibility:
   {', '.join(top_assets)}

2. Assets with the lowest average topological irreversibility:
   {', '.join(low_assets)}

3. Average Spearman correlation between TTI and rolling volatility:
   {avg_corr:.3f}

4. Mean standardized excess of observed TTI relative to shuffled surrogates:
   {avg_z_shuffle:.3f}

5. Mean standardized excess of observed TTI relative to phase-randomized surrogates:
   {avg_z_phase:.3f}

Interpretation:
If the z-scores are persistently positive, the observed asymmetry is stronger than expected under randomized temporal structure.
If the correlation with volatility is positive, TTI may be capturing regime stress, market turbulence, or structural disequilibrium.
"""
    return txt

print(interpret_results(corr_table, null_summary))

## 14. Export results
These files are useful for drafting tables and figures for a paper.

In [ ]:

prices_valid.to_csv(f"{OUTPUT_DIR}/prices_valid.csv")
returns_valid.to_csv(f"{OUTPUT_DIR}/returns_valid.csv")
tti_panel.to_csv(f"{OUTPUT_DIR}/tti_panel.csv")
corr_table.to_csv(f"{OUTPUT_DIR}/tti_volatility_summary.csv", index=False)
ranking.to_csv(f"{OUTPUT_DIR}/tti_cross_asset_ranking.csv", index=False)
null_table.to_csv(f"{OUTPUT_DIR}/tti_null_model_results.csv", index=False)
null_summary.to_csv(f"{OUTPUT_DIR}/tti_null_model_summary.csv")

print("Files exported to:", OUTPUT_DIR)

## 15. Paper-oriented next steps
To convert this exploration into a strong article, the next stage should include:

### Methodological strengthening
- Compare **TTI** against at least one standard irreversibility benchmark.
- Explore robustness across:
  - window length,
  - embedding dimension,
  - delay parameter,
  - asset frequency (daily vs intraday).

### Empirical strengthening
- Build event studies around:
  - COVID crash,
  - tightening cycles,
  - geopolitical shocks.
- Test whether TTI predicts:
  - future volatility,
  - drawdowns,
  - crisis-state probabilities.

### Statistical strengthening
- Use panel regressions:
  - TTI on lagged volatility, returns, VIX proxy, crisis dummies.
- Build asset-class comparisons with bootstrap confidence intervals.

### Publication strengthening
A strong paper title could be:
**Topological Time Irreversibility in Financial Markets: Persistent-Homology Evidence from Equities, FX, Crypto, and Commodities**